In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

DATA_PATH = "../processed/merged_daily.csv"

df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

df[["date", "pm2_5"]].head()

,date,pm2_5
0,2023-01-01 00:00:00+07:00,29.00
1,2023-01-02 00:00:00+07:00,35.54
2,2023-01-03 00:00:00+07:00,27.82
3,2023-01-04 00:00:00+07:00,19.46
4,2023-01-05 00:00:00+07:00,11.28


In [2]:
# Compute Z-score for PM2.5
pm_mean = df["pm2_5"].mean()
pm_std  = df["pm2_5"].std(ddof=0)

df["pm2_5_zscore"] = (df["pm2_5"] - pm_mean) / pm_std

# Threshold for anomalies (commonly 2 or 3 standard deviations)
z_thresh = 2
df["pm2_5_anom_z"] = df["pm2_5_zscore"].abs() > z_thresh
print(pm_mean, pm_std)

df[["date", "pm2_5", "pm2_5_zscore", "pm2_5_anom_z"]].head(10)

24.133780821917814 8.53946916679199


,date,pm2_5,pm2_5_zscore,pm2_5_anom_z
0,2023-01-01 00:00:00+07:00,29.00,0.569850,False
1,2023-01-02 00:00:00+07:00,35.54,1.335706,False
2,2023-01-03 00:00:00+07:00,27.82,0.431668,False
3,2023-01-04 00:00:00+07:00,19.46,-0.547315,False
4,2023-01-05 00:00:00+07:00,11.28,-1.505220,False
5,2023-01-06 00:00:00+07:00,12.00,-1.420906,False
6,2023-01-07 00:00:00+07:00,12.11,-1.408024,False
7,2023-01-08 00:00:00+07:00,26.40,0.265382,False
8,2023-01-09 00:00:00+07:00,36.39,1.435244,False
9,2023-01-10 00:00:00+07:00,9.94,-1.662139,False


In [3]:
# IQR-based anomaly detection for PM2.5
q1 = df["pm2_5"].quantile(0.25)
q3 = df["pm2_5"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

df["pm2_5_anom_iqr"] = (df["pm2_5"] < lower_bound) | (df["pm2_5"] > upper_bound)

print("Q1 =", q1, "Q3 =", q3, "IQR =", iqr)
print("Lower bound:", lower_bound, "Upper bound:", upper_bound)

df[["date", "pm2_5", "pm2_5_anom_iqr"]].head(10)

Q1 = 17.6 Q3 = 28.76 IQR = 11.16
Lower bound: 0.8599999999999994 Upper bound: 45.5


,date,pm2_5,pm2_5_anom_iqr
0,2023-01-01 00:00:00+07:00,29.00,False
1,2023-01-02 00:00:00+07:00,35.54,False
2,2023-01-03 00:00:00+07:00,27.82,False
3,2023-01-04 00:00:00+07:00,19.46,False
4,2023-01-05 00:00:00+07:00,11.28,False
5,2023-01-06 00:00:00+07:00,12.00,False
6,2023-01-07 00:00:00+07:00,12.11,False
7,2023-01-08 00:00:00+07:00,26.40,False
8,2023-01-09 00:00:00+07:00,36.39,False
9,2023-01-10 00:00:00+07:00,9.94,False


In [4]:
# Count how many days are flagged as anomalies
n_total = len(df)
n_z  = df["pm2_5_anom_z"].sum()
n_iqr = df["pm2_5_anom_iqr"].sum()

print(f"Total number of days: {n_total}")
print(f"Anomalous days by Z-score: {n_z} ({n_z/n_total*100:.2f}%)")
print(f"Anomalous days by IQR    : {n_iqr} ({n_iqr/n_total*100:.2f}%)")

Total number of days: 365
Anomalous days by Z-score: 17 (4.66%)
Anomalous days by IQR    : 8 (2.19%)


In [5]:
# Visualize anomalies based on Z-score
df_z = df.copy()
fig = px.line(df_z, x="date", y="pm2_5", title="PM2.5 with anomalies (Z-score)")

fig.add_scatter(
    x=df_z.loc[df_z["pm2_5_anom_z"], "date"],
    y=df_z.loc[df_z["pm2_5_anom_z"], "pm2_5"],
    mode="markers",
    marker=dict(color="red", size=10),
    name="Anomaly (Z-score)"
)

fig.update_layout(yaxis_title="PM2.5 (µg/m³)", xaxis_title="Date", template="plotly_white")
fig.show()

In [6]:
# Visualize anomalies based on IQR
df_iqr = df.copy()
fig = px.line(df_iqr, x="date", y="pm2_5", title="PM2.5 with anomalies (IQR)")

fig.add_scatter(
    x=df_iqr.loc[df_iqr["pm2_5_anom_iqr"], "date"],
    y=df_iqr.loc[df_iqr["pm2_5_anom_iqr"], "pm2_5"],
    mode="markers",
    marker=dict(color="orange", size=10),
    name="Anomaly (IQR)"
)

fig.update_layout(yaxis_title="PM2.5 (µg/m³)", xaxis_title="Date", template="plotly_white")
fig.show()